# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, following the Croissant data packaging specification. 

### Dataset Source
The dataset is provided using a Croissant schema (JSON-LD) at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant metadata and explore the general dataset description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets and their fields, referencing each entity by its `@id`.

We first enumerate all record sets. For each, we display its `@id`, name, description, and its fields' `@id`s.

In [ ]:
# Examine record sets and their fields using the mlcroissant metadata interface.
record_sets = list(dataset.record_sets)

print('Available Record Sets:')
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '<none>')}")
    print(f"  description: {rs.get('description', '<none>')}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print(f"  Fields:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - @id: {field.get('@id', '<unknown>')} | name: {field.get('name', '<none>')}")
            else:
                print(f"    - @id: {field}")
    print("")

## 3. Data Extraction
Extract data from each available record set. You should reference record sets and fields using their `@id`.

This will load each record set into a pandas DataFrame for subsequent exploration.

In [ ]:
# Collect the @id of all available record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Extracting data from Record Set: {record_set_id}")
    # Obtain records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"- Columns: {df.columns.tolist()}")
        print(df.head(2).to_string(index=False))
    else:
        print("- No data records found.")
    print("")

## 4. Exploratory Data Analysis (EDA)
Now, let's select a record set with data and perform filtering, normalization, and grouping, using fields referenced by their `@id`s.

> **Tip:** Replace `<record_set_id>`, `<numeric_field_id>`, `<group_field_id>` with the appropriate `@id`s based on the Data Overview.

In [ ]:
# For demonstration, we will use the first record set with valid data.
# Please adjust these variables based on your data overview:
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    display_df = dataframes[example_record_set_id]

    print(f"Using record set: {example_record_set_id}")

    # Inspect available columns (all referenced by @id)
    print('Columns (@id):', display_df.columns.tolist())

    # Attempt to identify a numeric field by @id
    numeric_candidate = None
    for col in display_df.columns:
        if pd.api.types.is_numeric_dtype(display_df[col]):
            numeric_candidate = col
            break
    if numeric_candidate is None:
        print('No obvious numeric field found.')
    else:
        numeric_field_id = numeric_candidate
        print(f'Numeric field selected: {numeric_field_id}')

        # Filter records where value > threshold
        threshold = display_df[numeric_field_id].mean()
        filtered_df = display_df[display_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric column
        colnorm = f"{numeric_field_id}_normalized"
        filtered_df[colnorm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, colnorm]].head())

        # Try grouping by another field
        group_field_id = None
        non_numeric_cols = [col for col in filtered_df.columns if not pd.api.types.is_numeric_dtype(filtered_df[col])]
        if non_numeric_cols:
            group_field_id = non_numeric_cols[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("Grouped mean values:")
            print(grouped_df.head())
        else:
            print('No non-numeric field found for grouping.')
else:
    print('No record sets with tabular data available.')

## 5. Visualization
Visualize the distribution of a numeric field or relationship, referencing columns by `@id`.

> **Note:** Adjust the plot to the available data and columns as shown in previous steps.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    df = dataframes[example_record_set_id]

    # Try to choose a numeric column by @id for plotting
    numeric_col = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_col = col
            break
    if numeric_col:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_col].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_col}")
        plt.xlabel(numeric_col)
        plt.ylabel('Count')
        plt.show()
    else:
        print('No numeric columns found for visualization.')
else:
    print('No data to visualize.')

## 6. Conclusion
In this notebook, we explored the FAIR² dataset using the `mlcroissant` library. We demonstrated how to load metadata and records, reference all entities by their `@id`, perform basic data cleaning and grouping, and visualize results. 

Please adjust the variable selections and plot configurations depending on the actual record sets and fields discovered in your data overview (Section 2).